In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [232]:
first_round = pd.read_csv("datos/classification first round/classification_final.csv")
second_round = pd.read_csv("datos/classification second_round/classification_secondRound_final.csv")

classifiers_first = pd.read_csv("datos/classification first round/classifiers_final.csv")
classifiers_second = pd.read_csv("datos/classification second_round/classifiers_secondRound_final.csv")

In [233]:
first_round = first_round[['id', 'classifier_id', 'ad_id', 'classification', 'ease', 'workday', 'timestamp']]

## Classifiers

Two classifiers used different email in both classification rounds
- magdalena.arismendi@mail.udp.cl = magda.arismendi@gmail.com
- mjarayaf@uc.cl = matilde.arfu@gmail.com

To identify them we use just the first email that was used in the firs classification

In [234]:
classifiers_second['email'][14] = 'mjarayaf@uc.cl'
classifiers_second['email'][3] = 'magdalena.arismendi@mail.udp.cl'

/tmp/ipykernel_42495/1207430555.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  classifiers_second['email'][14] = 'mjarayaf@uc.cl'
/tmp/ipykernel_42495/1207430555.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  classifiers_second['email'][3] = 'magdalena.arismendi@mail.udp.cl'


## Classifications

In [235]:
first_round['classification'].value_counts()

classification
Not WFH    11120
WFH         4880
Name: count, dtype: int64

In [236]:
first_round['ease'].value_counts()

ease
easy      12264
medium     2976
hard        760
Name: count, dtype: int64

In [237]:
first_round['workday'].value_counts()

workday
full         10454
not clear     3490
part          2056
Name: count, dtype: int64

In [238]:
second_round['classification'].value_counts()

classification
tot_remote          1546
not_remote           848
not_clear_remote     823
semi_remote          807
semi_presence        579
temp_remote          527
Name: count, dtype: int64

In [239]:
second_round['ease'].value_counts()

ease
easy      3985
medium     987
hard       158
Name: count, dtype: int64

In [240]:
classifications_m = pd.merge(first_round, second_round, on='ad_id', how='outer', suffixes=('_first', '_second'))

classifications agreement first round

In [241]:
unique_class_counts = first_round.groupby('ad_id')['classification'].nunique()

In [242]:
# Ads where both classifications are the same
same_classification = unique_class_counts[unique_class_counts == 1].count()

# Ads where the classifications differ
different_classification = unique_class_counts[unique_class_counts == 2].count()

print("Number of ads with same classification:", same_classification)
print("Number of ads with different classification:", different_classification)

Number of ads with same classification: 7750
Number of ads with different classification: 250


Indicator by classifier

In [243]:
# Step 1: For each ad, determine if the classifications agree or disagree
# We assume each ad appears exactly twice.
ad_agreement = first_round.groupby('ad_id')['classification'].nunique().reset_index()
ad_agreement['disagreement'] = ad_agreement['classification'].apply(lambda x: 0 if x == 1 else 1)

# Step 2: Merge the disagreement indicator back to the original DataFrame
df_first = first_round.merge(ad_agreement[['ad_id', 'disagreement']], on='ad_id', how='left')

# Step 3: For each classifier, compute:
#   - The total number of ads classified
#   - The sum of disagreements (i.e. the number of ads for which their classification did not match the other classifier's)
disagreement_by_classifier = df_first.groupby('classifier_id')['disagreement'].agg(total_ads='count', total_disagreements='sum')
disagreement_by_classifier['disagreement_rate'] = disagreement_by_classifier['total_disagreements'] / disagreement_by_classifier['total_ads']

In [244]:
disagreement_by_classifier = disagreement_by_classifier.sort_values(by='disagreement_rate', ascending=False)

In [245]:
disagreement_by_classifier = pd.merge(disagreement_by_classifier, classifiers_first, left_on='classifier_id', right_on='id', how='left')
disagreement_by_classifier.to_csv('datos/disagreement_by_classifier_first_round.csv')

Second round

In [246]:
unique_class_counts = second_round.groupby('ad_id')['classification'].nunique()

In [247]:
# Ads where both classifications are the same
same_classification = unique_class_counts[unique_class_counts == 1].count()

# Ads where the classifications differ
different_classification = unique_class_counts[unique_class_counts == 2].count()

print("Number of ads with same classification:", same_classification)
print("Number of ads with different classification:", different_classification)

Number of ads with same classification: 1801
Number of ads with different classification: 764


INdicator by classifier

In [248]:
# Step 1: For each ad, determine if the classifications agree or disagree
# We assume each ad appears exactly twice.
ad_agreement = second_round.groupby('ad_id')['classification'].nunique().reset_index()
ad_agreement['disagreement'] = ad_agreement['classification'].apply(lambda x: 0 if x == 1 else 1)

# Step 2: Merge the disagreement indicator back to the original DataFrame
df_second = second_round.merge(ad_agreement[['ad_id', 'disagreement']], on='ad_id', how='left')

# Step 3: For each classifier, compute:
#   - The total number of ads classified
#   - The sum of disagreements (i.e. the number of ads for which their classification did not match the other classifier's)
disagreement_by_classifier_second = df_second.groupby('classifier_id')['disagreement'].agg(total_ads='count', total_disagreements='sum')
disagreement_by_classifier_second['disagreement_rate'] = disagreement_by_classifier_second['total_disagreements'] / disagreement_by_classifier_second['total_ads']

In [249]:
disagreement_by_classifier = disagreement_by_classifier.sort_values(by='disagreement_rate', ascending=False)

In [250]:
disagreement_by_classifier_second = pd.merge(disagreement_by_classifier_second, classifiers_second, left_on='classifier_id', right_on='id', how='left')
disagreement_by_classifier_second.to_csv('datos/disagreement_by_classifier_second_round.csv')

All of the ads that receive at least one 'not_remote' in the second classification

In [251]:
len(classifications_m.loc[(classifications_m['classification_first'] == 'WFH') & (classifications_m['classification_second'] == 'not_remote')]['ad_id'].unique())

493

Ads with two WFH and two not_remote

In [252]:
# Group by ad_id and count occurrences for each condition
grouped = classifications_m.groupby('ad_id').agg(
    count_WFH = ('classification_first', lambda x: (x == 'WFH').sum()),
    count_not_remote = ('classification_second', lambda x: (x == 'not_remote').sum())
)

# Filter ad_ids that have exactly 2 "WFH" and 2 "not_remote"
selected_ads = grouped[(grouped['count_WFH'] == 2) & (grouped['count_not_remote'] == 2)]

# Get the number of such ad_ids
result = len(selected_ads)
print("Number of ad_ids with two 'WFH' and two 'not_remote':", result)

Number of ad_ids with two 'WFH' and two 'not_remote': 39


In [271]:
grouped = first_round.groupby('ad_id').agg(
    count_wfh = ('classification', lambda x: (x == 'WFH').sum())
)

In [253]:
first_round[first_round['classification'] == 'WFH']

,id,classifier_id,ad_id,classification,ease,workday,timestamp
15,15,6,3826451,WFH,hard,not clear,2024-07-14 19:17:06.597599
16,16,6,5423587,WFH,easy,not clear,2024-07-14 19:18:47.900291
17,17,6,4052452,WFH,easy,full,2024-07-14 19:19:35.489201
20,20,6,5298346,WFH,easy,full,2024-07-14 19:22:49.578868
29,29,6,4078590,WFH,easy,not clear,2024-07-14 19:28:48.432925
...,...,...,...,...,...,...,...
15988,15989,52,5637026,WFH,hard,full,2024-09-01 22:48:36.864955
15990,15991,49,4272781,WFH,easy,full,2024-09-01 22:49:42.234000
15993,15994,52,4056026,WFH,easy,full,2024-09-01 22:52:47.65054
15995,15996,52,4095979,WFH,easy,full,2024-09-01 22:55:13.311093
